# Data prep

In [ ]:
%load_ext autoreload
%autoreload 2

: 

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

In [ ]:

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

In [ ]:
from config import (
    PHENO_PATH,
    HC_LIST,
    MDD_LIST,
    SUBJ_LST_DIR,
)

In [ ]:
pheno_features = [
    "Primary_Dx",
    "Non-Primary_Dx",
    "Age",
    "Active Psychosis",
    "meds_yes_no",
    "live_with_whom",
    "house",
    "psych_meds",
]
pheno_df = pd.read_csv(PHENO_PATH, encoding="utf-8")

# search and replace column names with spaces or special characters
for col in pheno_df.columns:
    new_col = col.replace(" ", "_").replace("-", "_")
    pheno_df.rename(columns={col: new_col}, inplace=True)

# pheno_df.rename(columns={'Non-Primary_Dx': 'Non_Primary_Dx'}, inplace=True)

hc_df = pheno_df[(pheno_df.Primary_Dx=="999") & (pheno_df.Non_Primary_Dx=="999") & (pheno_df.Age < 100)]
mdd_df = pheno_df[(pheno_df.Primary_Dx=="MDD") & (pheno_df.Age < 100)]

# save list of subject IDs for HC and MDD groups
hc_df["subjectkey"].to_csv(os.path.join(SUBJ_LST_DIR, "no_Dx_subjects"), index=False, header=False)
mdd_df["subjectkey"].to_csv(os.path.join(SUBJ_LST_DIR, "mdd_primary_subjects"), index=False, header=False)

print(mdd_df)

# check if mdd_df is same as MDD_LIST
mdd_ids_from_df = set(mdd_df["subjectkey"].astype(str).tolist())
with open(MDD_LIST) as f:
    mdd_list = f.readlines()
    mdd_list = [line.strip() for line in mdd_list]
mdd_ids_from_list = set([str(i) for i in mdd_list])
print(mdd_ids_from_df)
print(mdd_ids_from_list)
print("MDD IDs match:", mdd_ids_from_df == mdd_ids_from_list)

# #  histogram of age distribution in hc_df and mdd_df
# plt.hist(hc_df["Age"], bins=40, alpha=0.5, label="HC")
# plt.hist(mdd_df["Age"], bins=40, alpha=0.5, label="MDD")
# plt.xlabel("Age")
# plt.ylabel("Count")
# plt.title("Age Distribution in HC and MDD Groups")
# plt.legend()
# plt.show()    

# for feature in pheno_features:
#     print(f"{feature}:")
#     print(pheno_df[feature].value_counts())
#     print("----------------------------------\n")

In [ ]:
def nearest_age_neighbor(target_age: int, candidates: pd.DataFrame) -> str:
    """
    Find the nearest age neighbor in candidates for the target subject and return the subjectkey.

    Args:
        target (int): Age in months of the target subject. 
        candidates (pd.DataFrame): DataFrame containing subject keys and their age in months.
    Returns:
        str: subjectkey of the nearest age neighbor.
    """
    age_diffs = candidates.iloc[:, 1] - target_age
    age_diffs = age_diffs.abs()
    nearest_idx = age_diffs.idxmin()
    return candidates.loc[nearest_idx, "subjectkey"]

def match_subjects_by_age(df1: pd.DataFrame, df2: pd.DataFrame) -> list[list[str, str]]:
    """
    Match subjects from df1 to df2 based on nearest age.

    Args:
        df1 (pd.DataFrame): DataFrame with subjects.
        df2 (pd.DataFrame): DataFrame with subjects to match against.
    Returns:
        list[list[str, str]]: List of tuples containing matched subjectkeys from df1 and df2.
        (Number of pairs equals the number of rows in the smaller DataFrame.)
    """
    matched_pairs = []

    if len(df1) > len(df2):
        age_df1 = df2[["subjectkey"]].copy()
        age_df1["age_months"] = df2["interview_age"] + (df2["Clin_admin_ndays"] % 30)
        age_df2 = df1[["subjectkey"]].copy()
        age_df2["age_months"] = df1["interview_age"] + (df1["Clin_admin_ndays"] % 30)
    else:
        age_df1 = df1[["subjectkey"]].copy()
        age_df1["age_months"] = df1["interview_age"] + (df1["Clin_admin_ndays"] % 30)
        age_df2 = df2[["subjectkey"]].copy()
        age_df2["age_months"] = df2["interview_age"] + (df2["Clin_admin_ndays"] % 30)

    for subject in age_df1.iloc:
        print(subject)
        nearest_subjkey = nearest_age_neighbor(subject.age_months, age_df2)
        matched_pairs.append([subject.subjectkey, nearest_subjkey])
        age_df2 = age_df2[age_df2.subjectkey != nearest_subjkey]  # remove matched subject

    return matched_pairs


def save_subject_pairs(pairs: list[list[str, str]], file1, file2):
    with open(file1, "w") as f1, open(file2, "w") as f2:
        for subj1, subj2 in pairs:
            f1.write(f"{subj1}\n")
            f2.write(f"{subj2}\n")

In [ ]:
mdd_hc_pairs = match_subjects_by_age(mdd_df, hc_df)
save_subject_pairs(
    mdd_hc_pairs,
    os.path.join(SUBJ_LST_DIR, "x_matched_mdd_subjects.txt"),
    os.path.join(SUBJ_LST_DIR, "x_matched_hc_subjects.txt"),
)



In [ ]:
print(mdd_hc_pairs)

In [ ]:
# print age pairs
for pair in mdd_hc_pairs:
    print(mdd_df[mdd_df.subjectkey == pair[0]].interview_age.values[0], hc_df[hc_df.subjectkey == pair[1]].interview_age.values[0])

# plot age pairs against age=age line

plt.figure(figsize=(8, 8))
ages_mdd = []
ages_hc = []
for pair in mdd_hc_pairs:
    ages_mdd.append(mdd_df[mdd_df.subjectkey == pair[0]].interview_age.values[0])
    ages_hc.append(hc_df[hc_df.subjectkey == pair[1]].interview_age.values[0])
plt.scatter(ages_mdd, ages_hc, alpha=0.6)
plt.plot([0, 800], [0, 800], color='red', linestyle='--')  # age=age line
plt.xlabel("MDD Age (months)")